# z022 – Importancia de variables del modelo final

**Idea:** entrenar UN modelo igual a los del ensamble final (trial 34 de z005, target BAJA+1 + BAJA+2, lags y deltas 1 y 2, train 202103-202106) y ver qué variables usó más.

**Importancia por "gain"** = cuánto mejoró el modelo gracias a los cortes en cada variable. Ejemplo: si `ctrx_quarter` tiene 20%, una quinta parte de todo lo que aprendió el modelo vino de esa variable.

Tarda ~3 min. No modifica ningún otro archivo.

In [1]:
# --- 1. Librerías y parámetros (iguales al submit 15) ---
import time
import pandas as pd
import lightgbm as lgb

SEMILLA     = 270131                       # una de las semillas del submit 16
MESES_FINAL = [202103, 202104, 202105, 202106]

params_fijos = {
    "objective": "binary", "learning_rate": 0.005, "min_data_in_leaf": 0,
    "feature_fraction": 0.5, "feature_fraction_bynode": 0.2, "max_bin": 31, "verbose": -1,
}
elegidos = pd.read_csv("z005_top10_final.csv", sep=";", decimal=",",
                       encoding="utf-8-sig", index_col="trial")
r = elegidos.loc[34]                        # el mejor trial real
params = {**params_fijos, "num_leaves": int(r["params_num_leaves"]),
          "min_sum_hessian_in_leaf": float(r["params_min_sum_hessian_in_leaf"]), "seed": SEMILLA}
params

{'objective': 'binary',
 'learning_rate': 0.005,
 'min_data_in_leaf': 0,
 'feature_fraction': 0.5,
 'feature_fraction_bynode': 0.2,
 'max_bin': 31,
 'verbose': -1,
 'num_leaves': 137,
 'min_sum_hessian_in_leaf': 9.755574121446362,
 'seed': 270131}

In [2]:
# --- 2. Datos, target juntar (BAJA+1 + BAJA+2) y lags/deltas (igual al submit 15) ---
df = pd.read_parquet("../competencia_01.parquet")
df["clase_binaria2"] = df["clase_ternaria"].isin(["BAJA+1", "BAJA+2"]).astype(int)

no_usar = ["foto_mes", "clase_ternaria", "clase_binaria2"]
cols = [c for c in df.columns if c not in no_usar and c != "numero_de_cliente"]

df = df.sort_values(["numero_de_cliente", "foto_mes"]).reset_index(drop=True)
grupo = df.groupby("numero_de_cliente")[cols]
nuevas = []
for k in [1, 2]:
    lag = grupo.shift(k).astype("float32")
    delta = (df[cols] - lag).astype("float32")
    lag.columns   = [f"{c}_lag{k}" for c in cols]
    delta.columns = [f"{c}_delta{k}" for c in cols]
    nuevas += [lag, delta]
df = pd.concat([df] + nuevas, axis=1)
del nuevas, lag, delta, grupo

variables = [c for c in df.columns if c not in no_usar]
final = df[df["foto_mes"].isin(MESES_FINAL)]
print("Variables:", len(variables), "| positivos:", final["clase_binaria2"].sum())

Variables: 761 | positivos: 8067


In [3]:
# --- 3. Entrenar un modelo (~2-3 min) ---
inicio = time.time()
datos = lgb.Dataset(final[variables], label=final["clase_binaria2"])
modelo = lgb.train(params, datos, num_boost_round=int(r["user_attrs_best_iter"]))
print(f"Listo en {(time.time() - inicio) / 60:.1f} min")

Listo en 2.7 min


In [4]:
# --- 4. Top 30 variables ---
imp = pd.DataFrame({"variable": variables,
                    "gain": modelo.feature_importance(importance_type="gain")})
imp["pct"] = (imp["gain"] / imp["gain"].sum() * 100).round(2)
imp = imp.sort_values("gain", ascending=False).reset_index(drop=True)
imp.index += 1                                # puesto 1, 2, 3...

# Tipo de variable: original, lag o delta (sirve para ver cuánto aportó el feature engineering)
imp["tipo"] = "original"
imp.loc[imp["variable"].str.contains("_lag"), "tipo"] = "lag"
imp.loc[imp["variable"].str.contains("_delta"), "tipo"] = "delta"

imp.to_csv("z022_importancia.csv", sep=";", decimal=",", encoding="utf-8-sig", index_label="puesto")
imp.head(30)

,variable,gain,pct,tipo
1,ctrx_quarter,185801.458208,5.15,original
2,mpasivos_margen,162330.359581,4.50,original
3,cpayroll_trx,157776.953837,4.37,original
4,mcuentas_saldo,152608.003856,4.23,original
5,mcaja_ahorro,149032.363134,4.13,original
6,mpayroll,134944.190348,3.74,original
7,mtarjeta_visa_consumo,118225.766149,3.28,original
8,ctarjeta_visa_transacciones,104459.897450,2.90,original
9,mautoservicio,97031.242910,2.69,original
10,ctarjeta_debito_transacciones,81061.297680,2.25,original


In [5]:
# --- 5. Resumen para la presentación ---
print("Aporte por tipo de variable (% del gain total):")
print(imp.groupby("tipo")["pct"].sum().round(1).sort_values(ascending=False))
print()
print("Lags/deltas dentro del top 30:", (imp.head(30)["tipo"] != "original").sum())
print("Variables que el modelo no usó nunca:", (imp["gain"] == 0).sum(), "de", len(imp))
print("Puesto de numero_de_cliente:", imp.index[imp["variable"] == "numero_de_cliente"][0])

Aporte por tipo de variable (% del gain total):
tipo
original    73.0
lag         16.8
delta       10.1
Name: pct, dtype: float64

Lags/deltas dentro del top 30: 5
Variables que el modelo no usó nunca: 167 de 761
Puesto de numero_de_cliente: 94
